In [0]:
-- ═══════════════════════════════════════════════════════════
-- Q1: Monthly GMV Trend
-- Business question: How is our gross merchandise value
-- trending month over month?
-- ═══════════════════════════════════════════════════════════
SELECT
    d.year,
    d.month_num,
    d.month_name,
    d.quarter_label,
    COUNT(DISTINCT f.order_id)              AS total_orders,
    COUNT(*)                                AS total_line_items,
    ROUND(SUM(f.item_price), 2)             AS gmv,
    ROUND(SUM(f.freight_value), 2)          AS total_freight,
    ROUND(SUM(f.total_item_value), 2)       AS total_revenue,
    ROUND(AVG(f.item_price), 2)             AS avg_item_price,
    ROUND(AVG(f.total_payment_value), 2)    AS avg_order_value
FROM paymentdw.gold.fact_orders f
JOIN paymentdw.gold.dim_date d
    ON f.date_key = d.date_key
WHERE f.order_status NOT IN ('cancelled')
GROUP BY d.year, d.month_num, d.month_name, d.quarter_label
ORDER BY d.year, d.month_num;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- Q2: Top 10 Product Categories by Revenue
-- Business question: Which categories drive the most revenue
-- and which have the best avg order value?
-- ═══════════════════════════════════════════════════════════

SELECT
    p.category,
    COUNT(DISTINCT f.order_id)              AS total_orders,
    COUNT(f.order_item_id)                  AS units_sold,
    ROUND(SUM(f.item_price), 2)             AS total_revenue,
    ROUND(AVG(f.item_price), 2)             AS avg_item_price,
    ROUND(AVG(f.review_score), 2)           AS avg_review_score,
    ROUND(AVG(f.delivery_days), 1)          AS avg_delivery_days
FROM paymentdw.gold.fact_orders f
JOIN paymentdw.gold.dim_product p
    ON f.product_key = p.product_key
WHERE f.order_status = 'delivered'
GROUP BY p.category
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- Q3: Seller Performance Scorecard
-- Business question: Who are our top and bottom sellers
-- across revenue, ratings, and delivery performance?
-- ═══════════════════════════════════════════════════════════
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT f.order_id)              AS total_orders,
    ROUND(SUM(f.item_price), 2)             AS total_revenue,
    ROUND(AVG(f.item_price), 2)             AS avg_item_price,
    ROUND(AVG(f.review_score), 2)           AS avg_review_score,
    ROUND(AVG(f.delivery_days), 1)          AS avg_delivery_days,
    SUM(CASE WHEN f.is_delivered_on_time 
             THEN 1 ELSE 0 END)             AS on_time_deliveries,
    COUNT(*)                                AS total_deliveries,
    ROUND(SUM(CASE WHEN f.is_delivered_on_time 
                   THEN 1 ELSE 0 END) * 100.0 
          / NULLIF(COUNT(*), 0), 2)         AS on_time_pct
FROM paymentdw.gold.fact_orders f
JOIN paymentdw.gold.dim_seller s
    ON f.seller_key = s.seller_key
WHERE f.order_status = 'delivered'
GROUP BY s.seller_id, s.seller_city, s.seller_state
HAVING COUNT(DISTINCT f.order_id) >= 10
ORDER BY total_revenue DESC;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- Q4: Payment Method Analysis
-- Business question: Which payment methods are most popular
-- and do installment payments correlate with higher order values?
-- ═══════════════════════════════════════════════════════════
SELECT
    pm.payment_type_label,
    pm.payment_category,
    COUNT(DISTINCT f.order_id)              AS total_orders,
    ROUND(SUM(f.total_payment_value), 2)    AS total_payment_volume,
    ROUND(AVG(f.total_payment_value), 2)    AS avg_order_value,
    ROUND(AVG(f.max_installments), 1)       AS avg_installments,
    SUM(CASE WHEN f.max_installments > 1 
             THEN 1 ELSE 0 END)             AS installment_orders,
    ROUND(SUM(CASE WHEN f.max_installments > 1 
                   THEN 1 ELSE 0 END) * 100.0
          / NULLIF(COUNT(*), 0), 2)         AS installment_pct,
    ROUND(AVG(f.review_score), 2)           AS avg_review_score
FROM paymentdw.gold.fact_orders f
JOIN paymentdw.gold.dim_payment_method pm
    ON f.payment_method_key = pm.payment_method_key
GROUP BY pm.payment_type_label, pm.payment_category
ORDER BY total_payment_volume DESC;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- Q5: Delivery Performance by State Corridor
-- Business question: Which seller → customer state corridors
-- have the worst delivery performance?
-- ═══════════════════════════════════════════════════════════
SELECT
    s.seller_state,
    c.customer_state,
    CONCAT(s.seller_state, ' → ', c.customer_state) AS corridor,
    COUNT(DISTINCT f.order_id)                      AS total_orders,
    ROUND(AVG(f.delivery_days), 1)                  AS avg_delivery_days,
    ROUND(AVG(CASE WHEN NOT f.is_delivered_on_time
                   THEN f.delivery_days END), 1)    AS avg_late_days,
    SUM(CASE WHEN NOT f.is_delivered_on_time
             THEN 1 ELSE 0 END)                     AS late_deliveries,
    ROUND(SUM(CASE WHEN NOT f.is_delivered_on_time
                   THEN 1 ELSE 0 END) * 100.0
          / NULLIF(COUNT(*), 0), 2)                 AS late_pct
FROM paymentdw.gold.fact_orders f
JOIN paymentdw.gold.dim_seller s
    ON f.seller_key = s.seller_key
JOIN paymentdw.gold.dim_customer c
    ON f.customer_key = c.customer_key
WHERE f.order_status = 'delivered'
  AND f.delivery_days IS NOT NULL
GROUP BY s.seller_state, c.customer_state
HAVING COUNT(DISTINCT f.order_id) >= 50
ORDER BY late_pct DESC
LIMIT 20;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- Q6: Transaction Error & Fraud Signal Analysis
-- Business question: What is our payment failure rate
-- by error type and card brand?
-- ═══════════════════════════════════════════════════════════
SELECT
    ft.error_type,
    ft.card_brand,
    ft.card_type,
    ft.payment_channel,
    COUNT(*)                                AS total_transactions,
    SUM(CASE WHEN ft.has_error 
             THEN 1 ELSE 0 END)             AS error_count,
    ROUND(SUM(CASE WHEN ft.has_error 
                   THEN 1 ELSE 0 END) * 100.0
          / NULLIF(COUNT(*), 0), 4)         AS error_rate_pct,
    ROUND(SUM(ft.amount), 2)                AS total_amount,
    ROUND(AVG(ft.amount), 2)                AS avg_transaction_amount
FROM paymentdw.gold.fact_transactions ft
WHERE ft.error_type != 'No Error'
GROUP BY ft.error_type, ft.card_brand, ft.card_type, ft.payment_channel
ORDER BY error_count DESC
LIMIT 20;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- Q7: Customer Cohort Analysis — Monthly Spend Retention
-- Business question: How does customer spending evolve
-- in months 1, 2, 3 after their first transaction?
-- ═══════════════════════════════════════════════════════════

WITH first_txn AS (
    SELECT
        client_id,
        DATE_TRUNC('month',
            TO_DATE(CAST(MIN(date_key) AS STRING), 'yyyyMMdd')) AS cohort_month
    FROM paymentdw.gold.fact_transactions
    WHERE client_id IS NOT NULL
    GROUP BY client_id
),
monthly_spend AS (
    SELECT
        ft.client_id,
        DATE_TRUNC('month',
            TO_DATE(CAST(ft.date_key AS STRING), 'yyyyMMdd'))   AS spend_month,
        SUM(ft.amount)                                          AS monthly_spend,
        COUNT(*)                                                AS txn_count
    FROM paymentdw.gold.fact_transactions ft
    WHERE ft.client_id IS NOT NULL
    GROUP BY ft.client_id, spend_month
)
SELECT
    DATE_FORMAT(f.cohort_month, 'yyyy-MM')              AS cohort,
    CAST(MONTHS_BETWEEN(ms.spend_month,
         f.cohort_month) AS INT)                        AS months_since_first,
    COUNT(DISTINCT ms.client_id)                        AS active_clients,
    ROUND(SUM(ms.monthly_spend), 2)                     AS total_spend,
    ROUND(AVG(ms.monthly_spend), 2)                     AS avg_spend_per_client,
    SUM(ms.txn_count)                                   AS total_transactions
FROM first_txn f
JOIN monthly_spend ms
    ON f.client_id = ms.client_id
WHERE CAST(MONTHS_BETWEEN(ms.spend_month,
      f.cohort_month) AS INT) BETWEEN 0 AND 5
GROUP BY cohort, months_since_first
ORDER BY cohort, months_since_first;

In [0]:
-- ═══════════════════════════════════════════════════════════
-- Q8: Dark Web Card Risk Analysis
-- Business question: What is the transaction volume exposure
-- from cards flagged on the dark web?
-- ═══════════════════════════════════════════════════════════
SELECT
    ft.card_on_dark_web,
    ft.card_brand,
    ft.card_type,
    COUNT(*)                                AS total_transactions,
    COUNT(DISTINCT ft.client_id)            AS affected_clients,
    ROUND(SUM(ft.amount), 2)                AS total_exposure,
    ROUND(AVG(ft.amount), 2)                AS avg_transaction_amount,
    SUM(CASE WHEN ft.has_error 
             THEN 1 ELSE 0 END)             AS error_transactions,
    ROUND(SUM(CASE WHEN ft.has_error 
                   THEN 1 ELSE 0 END) * 100.0
          / NULLIF(COUNT(*), 0), 2)         AS error_rate_pct
FROM paymentdw.gold.fact_transactions ft
WHERE ft.card_on_dark_web IS NOT NULL
GROUP BY ft.card_on_dark_web, ft.card_brand, ft.card_type
ORDER BY ft.card_on_dark_web DESC, total_exposure DESC;